# Modular Addition — Rebuttal Ablations

Compact 2×2 component ablations, train–test onset gaps, 100% follow-up, diagnostics, early stopping, and interruption-safe resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

P, FRAC_TRAIN = 113, 0.30
D_MODEL, NUM_HEADS, MLP_DIM = 128, 4, 512
LRS, SEEDS = [1e-4, 6e-4], list(range(1, 11))
LOG_EVERY, RECOVERY_EVERY = 50, 500
POST_99_GRACE, STABLE_100_LOGS = 1000, 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_ROOT = '/content/drive/MyDrive/grokking_rebuttal_addition'
os.makedirs(SAVE_ROOT, exist_ok=True)

RUN_COMPONENT_ABLATIONS = True
RUN_DIAGNOSTICS = True

CONFIGS = [
    # ---------------------------------------------------------
    # WD = 0
    # ---------------------------------------------------------
    dict(
        name="diag_sphere_bounded_lr_6e-4_wd0_long",
        model_type="spherical",
        bounded_head=True,
        tau=10.0,
        weight_decay=0.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),
    dict(
        name="diag_sphere_raw_lr_6e-4_wd0_long",
        model_type="spherical",
        bounded_head=False,
        tau=None,
        weight_decay=0.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),

    # ---------------------------------------------------------
    # WD = 1
    # ---------------------------------------------------------
    # Bu koşul zaten mevcut. Aynı isim sayesinde log + checkpoint
    # varsa otomatik olarak skip edilip summary'ye alınacak.
    dict(
        name="diag_sphere_bounded_lr_6e-4",
        model_type="spherical",
        bounded_head=True,
        tau=10.0,
        weight_decay=1.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),
    dict(
        name="diag_sphere_raw_lr_6e-4_wd1_long",
        model_type="spherical",
        bounded_head=False,
        tau=None,
        weight_decay=1.0,
        lr=6e-4,
        max_epochs=60_000,
        seeds=[1, 2, 3],
        instrument=True,
    ),
]


def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_dataset(p, frac_train, seed):
    set_seed(seed)
    pairs = [(a, b) for a in range(p) for b in range(p)]
    random.Random(seed).shuffle(pairs)
    n_train = int(len(pairs) * frac_train)
    def build(part):
        x = torch.tensor([[a, b, p] for a, b in part], dtype=torch.long)
        y = torch.tensor([(a + b) % p for a, b in part], dtype=torch.long)
        return x, y
    train_x, train_y = build(pairs[:n_train])
    test_x, test_y = build(pairs[n_train:])
    return train_x, train_y, test_x, test_y


class AdditionTransformer(nn.Module):
    def __init__(self, p, model_type, bounded_head, tau):
        super().__init__()
        self.model_type, self.bounded_head, self.tau = model_type, bounded_head, tau
        self.d_head = D_MODEL // NUM_HEADS
        self.tok_embed = nn.Embedding(p + 1, D_MODEL)
        self.pos_embed = nn.Embedding(3, D_MODEL)
        self.W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_O = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.mlp_in = nn.Linear(D_MODEL, MLP_DIM)
        self.mlp_out = nn.Linear(MLP_DIM, D_MODEL)
        self.unembed = nn.Linear(D_MODEL, p, bias=False)
        if model_type == 'layernorm':
            self.input_norm = nn.LayerNorm(D_MODEL)
            self.norm1 = nn.LayerNorm(D_MODEL)
            self.norm2 = nn.LayerNorm(D_MODEL)

    def norm(self, h, which):
        if self.model_type == 'spherical':
            return F.normalize(h, dim=-1)
        return self.input_norm(h) if which == 0 else (self.norm1(h) if which == 1 else self.norm2(h))

    def forward(self, x, return_diag=False):
        _, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h_embed_raw = self.tok_embed(x) + self.pos_embed(pos)
        h0 = self.norm(h_embed_raw, 0)

        Q = self.W_Q(h0).view(-1, L, NUM_HEADS, self.d_head).transpose(1, 2)
        K = self.W_K(h0).view(-1, L, NUM_HEADS, self.d_head).transpose(1, 2)
        V = self.W_V(h0).view(-1, L, NUM_HEADS, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        attn_out = self.W_O((attn @ V).transpose(1, 2).contiguous().view(-1, L, D_MODEL))

        attn_pre = h0 + attn_out
        h1 = self.norm(attn_pre, 1)
        mlp_pre = h1 + self.mlp_out(F.relu(self.mlp_in(h1)))
        h2 = self.norm(mlp_pre, 2)
        h_final = h2[:, 2, :]

        if self.bounded_head:
            logits = F.linear(h_final, F.normalize(self.unembed.weight, dim=1)) * float(self.tau)
        else:
            logits = self.unembed(h_final)

        if not return_diag:
            return logits

        eps = 1e-12
        uniform = torch.full_like(attn, 1.0 / L)
        entropy = -(attn.clamp_min(eps) * attn.clamp_min(eps).log()).sum(dim=-1)
        return logits, {
            'embed_raw_norm': h_embed_raw.norm(dim=-1).mean().item(),
            'input_post_norm': h0.norm(dim=-1).mean().item(),
            'attn_residual_pre_norm': attn_pre.norm(dim=-1).mean().item(),
            'attn_residual_post_norm': h1.norm(dim=-1).mean().item(),
            'mlp_residual_pre_norm': mlp_pre.norm(dim=-1).mean().item(),
            'mlp_residual_post_norm': h2.norm(dim=-1).mean().item(),
            'final_hidden_norm': h_final.norm(dim=-1).mean().item(),
            'logit_l2_norm': logits.norm(dim=-1).mean().item(),
            'max_abs_logit': logits.abs().amax(dim=-1).mean().item(),
            'attention_entropy': entropy.mean().item(),
            'attention_distance_to_uniform': (attn - uniform).pow(2).sum(dim=-1).sqrt().mean().item(),
            'unembed_row_norm_mean': self.unembed.weight.norm(dim=1).mean().item(),
            'unembed_row_norm_std': self.unembed.weight.norm(dim=1).std().item(),
        }


def parameter_norms(model):
    groups = {k: [] for k in ['global', 'embeddings', 'attention', 'mlp', 'unembedding']}
    for name, p in model.named_parameters():
        sq = p.detach().float().pow(2).sum()
        groups['global'].append(sq)
        if 'embed' in name: groups['embeddings'].append(sq)
        elif any(k in name for k in ['W_Q', 'W_K', 'W_V', 'W_O']): groups['attention'].append(sq)
        elif 'mlp_' in name: groups['mlp'].append(sq)
        elif 'unembed' in name: groups['unembedding'].append(sq)
    return {f'param_norm_{k}': math.sqrt(torch.stack(v).sum().item()) if v else 0.0 for k, v in groups.items()}


def first_onset(history, key, threshold):
    for epoch, value in zip(history['epochs'], history[key]):
        if value >= threshold:
            return epoch
    return None


def train_one(model, train_x, train_y, test_x, test_y, cfg, recovery_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'], betas=(0.9, 0.999))
    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)
    history = {k: [] for k in ['epochs', 'train_acc', 'test_acc', 'train_loss', 'test_loss']}
    diagnostics, start_epoch, test99_epoch, stable100 = [], 0, None, 0
    stop_reason = 'max_epochs'

    if os.path.exists(recovery_path):
        state = torch.load(recovery_path, map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        history, diagnostics = state['history'], state['diagnostics']
        start_epoch = state['epoch'] + 1
        test99_epoch, stable100 = state['test99_epoch'], state['stable100']
        print('resume:', cfg['name'], 'from', start_epoch)

    for epoch in tqdm(range(start_epoch, cfg['max_epochs']), desc=cfg['name'], leave=False):
        model.train()
        loss = criterion(model(train_x), train_y)
        if not torch.isfinite(loss):
            stop_reason = f'non_finite_loss_epoch_{epoch}'
            break
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()

        if epoch % LOG_EVERY != 0 and epoch != cfg['max_epochs'] - 1:
            continue

        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            if cfg['instrument']:
                test_logits, diag = model(test_x, return_diag=True)
                diag.update(parameter_norms(model)); diag['epoch'] = epoch
                diagnostics.append(diag)
            else:
                test_logits = model(test_x)
            train_loss = criterion(train_logits, train_y).item()
            test_loss = criterion(test_logits, test_y).item()
            train_acc = (train_logits.argmax(-1) == train_y).float().mean().item()
            test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

        for k, v in [('epochs', epoch), ('train_acc', train_acc), ('test_acc', test_acc),
                     ('train_loss', train_loss), ('test_loss', test_loss)]:
            history[k].append(v)

        if test99_epoch is None and test_acc >= 0.99:
            test99_epoch = epoch
        stable100 = stable100 + 1 if test_acc >= 1.0 - 1e-12 else 0

        if epoch % RECOVERY_EVERY == 0:
            torch.save(dict(epoch=epoch, model_state_dict=model.state_dict(),
                            optimizer_state_dict=optimizer.state_dict(), history=history,
                            diagnostics=diagnostics, test99_epoch=test99_epoch,
                            stable100=stable100), recovery_path)

        if stable100 >= STABLE_100_LOGS:
            stop_reason = 'stable_test_100'
            break
        if test99_epoch is not None and epoch >= test99_epoch + POST_99_GRACE:
            stop_reason = 'post_99_grace_complete'
            break

    return history, diagnostics, stop_reason


def make_summary(cfg, seed, history, stop_reason):
    out = dict(config=cfg['name'], seed=seed, model_type=cfg['model_type'],
               bounded_head=cfg['bounded_head'], tau=cfg['tau'],
               weight_decay=cfg['weight_decay'], lr=cfg['lr'],
               max_epochs=cfg['max_epochs'], stop_reason=stop_reason,
               last_epoch=history['epochs'][-1] if history['epochs'] else None,
               peak_test_acc=max(history['test_acc']) if history['test_acc'] else None)
    for label, threshold in [('95', 0.95), ('99', 0.99), ('100', 1.0 - 1e-12)]:
        tr = first_onset(history, 'train_acc', threshold)
        te = first_onset(history, 'test_acc', threshold)
        out[f'train_epoch_{label}'] = tr
        out[f'test_epoch_{label}'] = te
        out[f'gap_{label}'] = None if tr is None or te is None else te - tr
    return out


rows = []
for cfg in CONFIGS:
    cfg_dir = os.path.join(SAVE_ROOT, cfg['name'])
    os.makedirs(cfg_dir, exist_ok=True)
    for seed in cfg['seeds']:
        stem = f"{cfg['name']}_seed{seed}"
        log_path = os.path.join(cfg_dir, stem + '.json')
        ckpt_path = os.path.join(cfg_dir, stem + '.pt')
        recovery_path = os.path.join(cfg_dir, stem + '_recovery.pt')

        if os.path.exists(log_path) and os.path.exists(ckpt_path):
            with open(log_path) as f: saved = json.load(f)
            rows.append(saved['summary']); print('skip:', stem); continue

        train_x, train_y, test_x, test_y = make_dataset(P, FRAC_TRAIN, seed)
        set_seed(seed)
        model = AdditionTransformer(P, cfg['model_type'], cfg['bounded_head'], cfg['tau']).to(DEVICE)
        history, diagnostics, stop_reason = train_one(
            model, train_x, train_y, test_x, test_y, cfg, recovery_path
        )
        summary = make_summary(cfg, seed, history, stop_reason)

        torch.save(dict(model_state_dict=model.state_dict(), summary=summary, config=cfg), ckpt_path)
        with open(log_path, 'w') as f:
            json.dump(dict(summary=summary, history=history, diagnostics=diagnostics), f, indent=2)
        if os.path.exists(recovery_path): os.remove(recovery_path)

        rows.append(summary)
        pd.DataFrame(rows).to_csv(os.path.join(SAVE_ROOT, 'summary_incremental.csv'), index=False)
        print('saved:', stem)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()


df = pd.DataFrame(rows)
df.to_csv(os.path.join(SAVE_ROOT, 'summary_final.csv'), index=False)
print('\nPer-run results:')
print(df.sort_values(['config', 'seed']).to_string(index=False))

agg = (df.groupby('config', dropna=False)
       .agg(seeds=('seed', 'count'),
            reached_95=('test_epoch_95', lambda x: x.notna().sum()),
            reached_99=('test_epoch_99', lambda x: x.notna().sum()),
            reached_100=('test_epoch_100', lambda x: x.notna().sum()),
            mean_test_95=('test_epoch_95', 'mean'), std_test_95=('test_epoch_95', 'std'),
            mean_test_99=('test_epoch_99', 'mean'), std_test_99=('test_epoch_99', 'std'),
            mean_gap_95=('gap_95', 'mean'), mean_gap_99=('gap_99', 'mean'),
            mean_peak_test_acc=('peak_test_acc', 'mean'))
       .reset_index())
agg.to_csv(os.path.join(SAVE_ROOT, 'configuration_summary.csv'), index=False)
print('\nConfiguration summary:')
print(agg.to_string(index=False))


## Fresh paired robustness sweep (70 runs)

This section is independent of the original 2×2×2 ablation above. It runs a fresh, paired one-factor-at-a-time robustness study:

- **Treatment:** spherical residual stream + bounded unembedding + WD=0
- **Matched control:** LayerNorm residual stream + the same bounded unembedding + WD=0
- **Order:** all spherical runs first, then all LayerNorm runs
- **Settings:** canonical, width, depth, modulus, train fraction, optimizer, and LR schedule
- **Seeds:** 5 per model/setting
- **Budget:** 60,000 epochs, logging every 200 epochs
- **Early stop:** three consecutive logged evaluations at 100% test accuracy
- **Resume:** interruption-safe recovery checkpoints stored on Drive

Run the Drive-mount cell near the top of the notebook, then run the three cells in this section. The original ablation cell does not need to be rerun.


In [ ]:
# ============================================================
# ROBUSTNESS SWEEP — EDITABLE SETTINGS
# ============================================================

# Self-contained imports: only the Drive-mount cell above is required.
import os, json, math, random, time
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

ROBUST_SAVE_ROOT = '/content/drive/MyDrive/grokking_rebuttal_addition_robustness_fresh'
os.makedirs(ROBUST_SAVE_ROOT, exist_ok=True)

ROBUST_SEEDS = [1, 2, 3, 4, 5]
ROBUST_MAX_EPOCHS = 60_000
ROBUST_LOG_EVERY = 200
ROBUST_RECOVERY_EVERY = 1_000
ROBUST_STABLE_100_LOGS = 3

# Canonical condition: the cleanest positive setting from the component ablation.
BASE = dict(
    p=113,
    frac_train=0.30,
    d_model=128,
    n_heads=4,
    mlp_dim=512,
    n_layers=1,
    optimizer='adamw',
    lr=6e-4,
    schedule='constant',
    min_lr_ratio=0.10,
    bounded_head=True,
    tau=10.0,
    weight_decay=0.0,
)

# One factor changes at a time. Keep this list explicit and easy to audit.
ROBUSTNESS_VARIANTS = [
    dict(name='canonical'),
    dict(name='width64', d_model=64, mlp_dim=256),
    dict(name='depth2', n_layers=2),
    dict(name='modulus97', p=97),
    dict(name='train_fraction20', frac_train=0.20),
    dict(name='optimizer_rmsprop', optimizer='rmsprop'),
    dict(name='schedule_cosine', schedule='cosine'),
]

# Sphere first, exactly as requested; matched LayerNorm controls follow.
MODEL_ORDER = ['spherical', 'layernorm']


def build_robust_configs():
    configs = []
    for model_type in MODEL_ORDER:
        for variant in ROBUSTNESS_VARIANTS:
            cfg = dict(BASE)
            cfg.update(variant)
            cfg.update(
                model_type=model_type,
                seeds=list(ROBUST_SEEDS),
                max_epochs=ROBUST_MAX_EPOCHS,
                log_every=ROBUST_LOG_EVERY,
                recovery_every=ROBUST_RECOVERY_EVERY,
                stable_100_logs=ROBUST_STABLE_100_LOGS,
            )
            cfg['run_name'] = f"{model_type}__{cfg['name']}"
            configs.append(cfg)
    return configs


ROBUST_CONFIGS = build_robust_configs()
print(f'{len(ROBUST_CONFIGS)} configurations × {len(ROBUST_SEEDS)} seeds = '
      f'{len(ROBUST_CONFIGS) * len(ROBUST_SEEDS)} runs')
print('Execution order:')
for cfg in ROBUST_CONFIGS:
    print('  ', cfg['run_name'])


In [ ]:
# ============================================================
# ROBUSTNESS SWEEP — MODEL, DATA, TRAINING, AND RESUME
# ============================================================


def atomic_torch_save(obj, path):
    """Write a checkpoint atomically so an interrupted write does not corrupt it."""
    tmp_path = path + '.tmp'
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)


def atomic_json_dump(obj, path):
    """Write JSON atomically for the same reason."""
    tmp_path = path + '.tmp'
    with open(tmp_path, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp_path, path)


def make_addition_dataset(p, frac_train, seed):
    set_seed(seed)
    pairs = [(a, b) for a in range(p) for b in range(p)]
    random.Random(seed).shuffle(pairs)
    n_train = int(len(pairs) * frac_train)

    def build(part):
        x = torch.tensor([[a, b, p] for a, b in part], dtype=torch.long)
        y = torch.tensor([(a + b) % p for a, b in part], dtype=torch.long)
        return x, y

    train_x, train_y = build(pairs[:n_train])
    test_x, test_y = build(pairs[n_train:])
    return train_x, train_y, test_x, test_y


class RobustTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_dim, model_type):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError(f'd_model={d_model} must be divisible by n_heads={n_heads}')

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.model_type = model_type

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)

        if model_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)

    def norm1_fn(self, h):
        return F.normalize(h, dim=-1) if self.model_type == 'spherical' else self.norm1(h)

    def norm2_fn(self, h):
        return F.normalize(h, dim=-1) if self.model_type == 'spherical' else self.norm2(h)

    def forward(self, h):
        batch_size, seq_len, _ = h.shape

        Q = self.W_Q(h).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        joined = (attn @ V).transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        h = self.norm1_fn(h + self.W_O(joined))
        h = self.norm2_fn(h + self.mlp_out(F.relu(self.mlp_in(h))))
        return h


class RobustAdditionTransformer(nn.Module):
    """Depth=1 exactly follows the architecture used in the original ablation cell."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = deepcopy(cfg)
        self.model_type = cfg['model_type']
        self.bounded_head = cfg['bounded_head']
        self.tau = cfg['tau']

        p = cfg['p']
        d_model = cfg['d_model']
        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        if self.model_type == 'layernorm':
            self.input_norm = nn.LayerNorm(d_model)

        self.blocks = nn.ModuleList([
            RobustTransformerBlock(
                d_model=d_model,
                n_heads=cfg['n_heads'],
                mlp_dim=cfg['mlp_dim'],
                model_type=self.model_type,
            )
            for _ in range(cfg['n_layers'])
        ])
        self.unembed = nn.Linear(d_model, p, bias=False)

    def input_norm_fn(self, h):
        return F.normalize(h, dim=-1) if self.model_type == 'spherical' else self.input_norm(h)

    def forward(self, x):
        seq_len = x.shape[1]
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self.input_norm_fn(self.tok_embed(x) + self.pos_embed(pos))

        for block in self.blocks:
            h = block(h)

        h_final = h[:, 2, :]
        if self.bounded_head:
            normalized_head = F.normalize(self.unembed.weight, dim=1)
            return F.linear(h_final, normalized_head) * float(self.tau)
        return self.unembed(h_final)


def make_optimizer(model, cfg):
    common = dict(lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    if cfg['optimizer'] == 'adamw':
        return torch.optim.AdamW(model.parameters(), betas=(0.9, 0.999), **common)
    if cfg['optimizer'] == 'rmsprop':
        return torch.optim.RMSprop(
            model.parameters(), alpha=0.99, eps=1e-8, momentum=0.0, **common
        )
    raise ValueError(f"Unknown optimizer: {cfg['optimizer']}")


def make_scheduler(optimizer, cfg):
    if cfg['schedule'] == 'constant':
        return None
    if cfg['schedule'] == 'cosine':
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg['max_epochs'],
            eta_min=cfg['lr'] * cfg['min_lr_ratio'],
        )
    raise ValueError(f"Unknown schedule: {cfg['schedule']}")


def first_logged_onset(history, key, threshold):
    for epoch, value in zip(history['epochs'], history[key]):
        if value >= threshold:
            return epoch
    return None


def make_robust_summary(cfg, seed, history, stop_reason, elapsed_seconds):
    summary = dict(
        run_name=cfg['run_name'],
        variant=cfg['name'],
        seed=seed,
        model_type=cfg['model_type'],
        p=cfg['p'],
        frac_train=cfg['frac_train'],
        d_model=cfg['d_model'],
        n_heads=cfg['n_heads'],
        mlp_dim=cfg['mlp_dim'],
        n_layers=cfg['n_layers'],
        optimizer=cfg['optimizer'],
        lr=cfg['lr'],
        schedule=cfg['schedule'],
        bounded_head=cfg['bounded_head'],
        tau=cfg['tau'],
        weight_decay=cfg['weight_decay'],
        max_epochs=cfg['max_epochs'],
        log_every=cfg['log_every'],
        stop_reason=stop_reason,
        elapsed_seconds=elapsed_seconds,
        last_epoch=history['epochs'][-1] if history['epochs'] else None,
        peak_test_acc=max(history['test_acc']) if history['test_acc'] else None,
    )

    for label, threshold in [('95', 0.95), ('99', 0.99), ('100', 1.0 - 1e-12)]:
        train_epoch = first_logged_onset(history, 'train_acc', threshold)
        test_epoch = first_logged_onset(history, 'test_acc', threshold)
        summary[f'train_epoch_{label}'] = train_epoch
        summary[f'test_epoch_{label}'] = test_epoch
        summary[f'gap_{label}'] = (
            None if train_epoch is None or test_epoch is None else test_epoch - train_epoch
        )

    return summary


def train_robust_run(model, data, cfg, recovery_path):
    train_x, train_y, test_x, test_y = [tensor.to(DEVICE) for tensor in data]
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg)

    history = {
        'epochs': [],
        'train_acc': [],
        'test_acc': [],
        'train_loss': [],
        'test_loss': [],
        'learning_rate': [],
    }
    start_epoch = 0
    stable_test_100 = 0
    stop_reason = 'max_epochs'

    if os.path.exists(recovery_path):
        state = torch.load(recovery_path, map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        if scheduler is not None and state.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(state['scheduler_state_dict'])
        history = state['history']
        stable_test_100 = state['stable_test_100']
        start_epoch = state['epoch'] + 1
        print(f"resume: {cfg['run_name']} seed={state['seed']} from epoch {start_epoch}")

    for epoch in tqdm(
        range(start_epoch, cfg['max_epochs']),
        desc=f"{cfg['run_name']}",
        leave=False,
    ):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(train_x), train_y)

        if not torch.isfinite(loss):
            stop_reason = f'non_finite_loss_epoch_{epoch}'
            break

        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        should_log = (epoch % cfg['log_every'] == 0) or (epoch == cfg['max_epochs'] - 1)
        if should_log:
            model.eval()
            with torch.no_grad():
                train_logits = model(train_x)
                test_logits = model(test_x)
                train_loss = criterion(train_logits, train_y).item()
                test_loss = criterion(test_logits, test_y).item()
                train_acc = (train_logits.argmax(-1) == train_y).float().mean().item()
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

            history['epochs'].append(epoch)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)
            history['train_loss'].append(train_loss)
            history['test_loss'].append(test_loss)
            history['learning_rate'].append(optimizer.param_groups[0]['lr'])

            stable_test_100 = stable_test_100 + 1 if test_acc >= 1.0 - 1e-12 else 0

            print(
                f"epoch={epoch:6d}  train={100*train_acc:6.2f}%  "
                f"test={100*test_acc:6.2f}%  lr={optimizer.param_groups[0]['lr']:.3e}"
            )

            if stable_test_100 >= cfg['stable_100_logs']:
                stop_reason = 'stable_test_100'
                break

        should_checkpoint = (
            epoch % cfg['recovery_every'] == 0
            or should_log and epoch == cfg['max_epochs'] - 1
        )
        if should_checkpoint:
            atomic_torch_save(
                dict(
                    epoch=epoch,
                    seed=cfg['_active_seed'],
                    model_state_dict=model.state_dict(),
                    optimizer_state_dict=optimizer.state_dict(),
                    scheduler_state_dict=None if scheduler is None else scheduler.state_dict(),
                    history=history,
                    stable_test_100=stable_test_100,
                    config=cfg,
                ),
                recovery_path,
            )

    return history, stop_reason


In [ ]:
# ============================================================
# ROBUSTNESS SWEEP — RUNNER AND FINAL TABLES
# ============================================================

robust_rows = []

for cfg_template in ROBUST_CONFIGS:
    cfg_dir = os.path.join(ROBUST_SAVE_ROOT, cfg_template['run_name'])
    os.makedirs(cfg_dir, exist_ok=True)

    for seed in cfg_template['seeds']:
        cfg = deepcopy(cfg_template)
        cfg['_active_seed'] = seed

        stem = f"{cfg['run_name']}__seed{seed}"
        json_path = os.path.join(cfg_dir, stem + '.json')
        model_path = os.path.join(cfg_dir, stem + '.pt')
        recovery_path = os.path.join(cfg_dir, stem + '__recovery.pt')

        # A completed run is never repeated. Partial runs resume from recovery_path.
        if os.path.exists(json_path) and os.path.exists(model_path):
            with open(json_path) as f:
                saved = json.load(f)
            robust_rows.append(saved['summary'])
            print('skip completed:', stem)
            continue

        print('\n' + '=' * 80)
        print('START', stem)
        print({k: v for k, v in cfg.items() if k not in ['seeds', '_active_seed']})

        data = make_addition_dataset(cfg['p'], cfg['frac_train'], seed)
        set_seed(seed)
        model = RobustAdditionTransformer(cfg).to(DEVICE)

        started = time.time()
        history, stop_reason = train_robust_run(model, data, cfg, recovery_path)
        elapsed_seconds = time.time() - started
        summary = make_robust_summary(cfg, seed, history, stop_reason, elapsed_seconds)

        atomic_torch_save(
            dict(model_state_dict=model.state_dict(), summary=summary, config=cfg),
            model_path,
        )
        atomic_json_dump(
            dict(summary=summary, history=history, config=cfg),
            json_path,
        )
        if os.path.exists(recovery_path):
            os.remove(recovery_path)

        robust_rows.append(summary)
        pd.DataFrame(robust_rows).to_csv(
            os.path.join(ROBUST_SAVE_ROOT, 'summary_incremental.csv'), index=False
        )
        print('SAVED', stem, '|', stop_reason)

        del model, data
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


robust_df = pd.DataFrame(robust_rows)
robust_df.to_csv(os.path.join(ROBUST_SAVE_ROOT, 'summary_final.csv'), index=False)

print('\nPer-run results:')
show_cols = [
    'model_type', 'variant', 'seed', 'stop_reason', 'peak_test_acc',
    'test_epoch_95', 'test_epoch_99', 'test_epoch_100',
    'gap_95', 'gap_99', 'gap_100', 'last_epoch',
]
print(robust_df[show_cols].sort_values(['model_type', 'variant', 'seed']).to_string(index=False))


def reached_count(series):
    return int(series.notna().sum())


robust_agg = (
    robust_df.groupby(['variant', 'model_type'], dropna=False)
    .agg(
        seeds=('seed', 'count'),
        reached_95=('test_epoch_95', reached_count),
        reached_99=('test_epoch_99', reached_count),
        reached_100=('test_epoch_100', reached_count),
        mean_test_95=('test_epoch_95', 'mean'),
        std_test_95=('test_epoch_95', 'std'),
        mean_test_99=('test_epoch_99', 'mean'),
        std_test_99=('test_epoch_99', 'std'),
        mean_test_100=('test_epoch_100', 'mean'),
        std_test_100=('test_epoch_100', 'std'),
        mean_gap_95=('gap_95', 'mean'),
        mean_gap_99=('gap_99', 'mean'),
        mean_gap_100=('gap_100', 'mean'),
        mean_peak_test_acc=('peak_test_acc', 'mean'),
    )
    .reset_index()
)
robust_agg.to_csv(os.path.join(ROBUST_SAVE_ROOT, 'configuration_summary.csv'), index=False)

print('\nConfiguration summary:')
print(robust_agg.to_string(index=False))

# A compact paired table for the rebuttal: sphere and matched LayerNorm side by side.
paired_99 = robust_agg.pivot(index='variant', columns='model_type', values='mean_test_99')
paired_hits = robust_agg.pivot(index='variant', columns='model_type', values='reached_99')
paired = pd.DataFrame(index=paired_99.index)
paired['sphere_mean_test99'] = paired_99.get('spherical')
paired['layernorm_mean_test99'] = paired_99.get('layernorm')
paired['sphere_hits_99'] = paired_hits.get('spherical')
paired['layernorm_hits_99'] = paired_hits.get('layernorm')
paired['layernorm_minus_sphere_test99'] = (
    paired['layernorm_mean_test99'] - paired['sphere_mean_test99']
)
paired = paired.reset_index()
paired.to_csv(os.path.join(ROBUST_SAVE_ROOT, 'paired_rebuttal_table.csv'), index=False)

print('\nPaired 99% test-accuracy table:')
print(paired.to_string(index=False))
